# RealVisXL V4.0 Uncensored Studio
Run this cell to launch the WebUI.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio

import torch
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
import gradio as gr

print("Loading RealVisXL V4.0...")
pipe_txt2img = AutoPipelineForText2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

pipe_img2img = AutoPipelineForImage2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

def generate_or_edit(image, prompt, negative_prompt, mode, strength, steps):
    if mode == "Text to Image":
        return pipe_txt2img(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=int(steps), guidance_scale=7.5).images[0]
    else:
        if image is None:
            return None
        return pipe_img2img(prompt=prompt, negative_prompt=negative_prompt, image=image, strength=float(strength), num_inference_steps=int(steps), guidance_scale=7.5).images[0]

demo = gr.Interface(
    fn=generate_or_edit,
    inputs=[
        gr.Image(type="pil", label="Source Image for Edit"),
        gr.Textbox(label="Prompt", value="Cinematic portrait, highly detailed"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality"),
        gr.Radio(["Text to Image", "Image to Image"], value="Text to Image", label="Mode"),
        gr.Slider(0.1, 0.9, value=0.5, label="Strength"),
        gr.Slider(10, 50, value=25, step=1, label="Steps")
    ],
    outputs=gr.Image(label="Output"),
    title="RealVisXL V4.0 Studio"
)
demo.launch(share=True, debug=True)